In [ ]:
import re
import time
from os import listdir, walk, makedirs
from os.path import isdir, join, exists
import pandas as pd
import swifter
from simplet5 import SimpleT5

In [ ]:
experiment_code = 'NBCODE'
version = NBVERSION
fold = NBFOLD
k = NBK
experiment_name = 'NBNAME'
model_name = 'NBMODEL'
dataset_path = '../NBDATASETNUMBER-NBDATASETNAME'
model_path = '../../models/' + model_name
best_epoch = TESTBEST

In [ ]:
dataset_basename = 'NBDATASETNUMBER-{}-prepared-dataset-{}'.format(experiment_code, experiment_name)
test_path = '{}/{}-test.csv'.format(dataset_path, dataset_basename)
print('test path: ' + test_path)

basename = 'NBNUMBER-{}-v{}-{}_{}-{}-{}'.format(experiment_code, version, fold, k, experiment_name, model_name)
models_path = '{}-1-training-outputs'.format(basename)
output_path = '{}-3-accuracy-test-best-predictions'.format(basename)

basename_accuracies = 'NBNUMBER-{}-v{}-{}-{}'.format(experiment_code, version, experiment_name, model_name)
accuracies_path = '{}-3-accuracy-test-best-predictions.csv'.format(basename_accuracies)

print('models path: ' + models_path)
print('output path: ' + output_path)
print('accuracies path: ' + accuracies_path)

In [ ]:
if not exists('{}'.format(output_path)):
    makedirs('{}'.format(output_path))

In [ ]:
df_test = pd.read_csv(test_path)
df_test['source_text'] = df_test['source_text'].astype(str)
df_test['target_text'] = df_test['target_text'].astype(str)

In [ ]:
df_test.head()

In [ ]:
test = df_test.copy()
test['target_text'].value_counts()

In [ ]:
models = [join(models_path, name) for name in listdir(models_path) if isdir(join(models_path, name))]
models.sort()
models

In [ ]:
best_model = models[best_epoch]
best_model

In [ ]:
start = time.time()
    
model_train_data = SimpleT5()
model_train_data.load_model('t5', best_model, use_gpu = False)
test['prediction'] = test['source_text'].swifter.apply(lambda x: model_train_data.predict(x)[0])

accuracy = test[test['prediction']==test['target_text']].shape[0]/test.shape[0]

test.to_csv('{}/epoch-{}-test.csv'.format(output_path, best_epoch), index=False)
test = test.drop('prediction', axis=1)

end = time.time()
t = end - start
print('Time: {} rows in {}s'.format(test.shape[0], t))

In [ ]:
if exists(accuracies_path):
    df_accuracies = pd.read_csv(accuracies_path, sep = ';', decimal= ',')
    df_accuracies = df_accuracies[df_accuracies['fold'] != fold]
else:
    df_accuracies = pd.DataFrame(columns=['experiment', 'fold', 'best_epoch', 'accuracy', 'time'])

df_accuracies.loc[-1] = [basename_accuracies, int(fold), best_epoch, accuracy, float(t)]
df_accuracies.sort_values(by=['fold'], inplace=True)
df_accuracies.to_csv(accuracies_path, index=False, sep = ';', decimal= ',', float_format='%.15f')

In [ ]:
print('epoch =', best_epoch)
print('accuracy =', accuracy)